# IGP: QLoRA Fine-tuning on Kaggle T4 GPU
**Karan — UWE Bristol MSc Data Science IGP**

Run cells top to bottom. GPU must be enabled (Settings > Accelerator > GPU T4 x2).

In [ ]:
# Step 1: Check GPU
import torch
print('GPU available:', torch.cuda.is_available())
print('GPU name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if torch.cuda.is_available() else 0)

In [ ]:
# Step 2: Install packages
# This takes 2-3 minutes on first run.
# peft and accelerate pinned (were >=): drift here is the leading suspect for the
# v2 adapter dropping the action label. Matched to transformers 4.45.2.
!pip install -q \
  "transformers==4.45.2" \
  "trl==0.11.4" \
  "peft==0.13.2" \
  "bitsandbytes==0.45.5" \
  "accelerate==0.34.2" \
  datasets
import peft, accelerate, transformers, trl
print('versions:', 'transformers', transformers.__version__, '| trl', trl.__version__,
      '| peft', peft.__version__, '| accelerate', accelerate.__version__)

In [ ]:
# Step 3: Upload your data files using the Kaggle sidebar
# Click the folder icon on the left > Upload > select sme_train_v2.jsonl and sme_val_v2.jsonl
# Then update these paths:
TRAIN_FILE = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_train_v2.jsonl'
VAL_FILE   = '/kaggle/input/datasets/karanhomayounfar1/sme-voice-data/sme_val_v2.jsonl'
OUTPUT_DIR = '/kaggle/working/checkpoints/sme-phi3-qlora-v2'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Paths set.')

In [ ]:
# Step 4: Load model + tokenizer
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'microsoft/Phi-3-mini-4k-instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True, padding_side='right')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# T4 is Turing architecture -- no flash_attention_2, use eager
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    attn_implementation='eager',
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print('Model loaded.')

In [ ]:
# Step 5: Apply QLoRA
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    bias='none',
    inference_mode=False,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Step 6: Load and format dataset
from datasets import load_dataset

def fmt(rec):
    return {'text': (
        f"<|system|>\n{rec['instruction']}<|end|>\n"
        f"<|user|>\n{rec['input']}<|end|>\n"
        f"<|assistant|>\n{rec['output']}<|end|>"
    )}

train_ds = load_dataset('json', data_files=TRAIN_FILE, split='train').map(fmt)
val_ds   = load_dataset('json', data_files=VAL_FILE,   split='train').map(fmt)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

In [ ]:
# Step 7: Train
from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM

collator = DataCollatorForCompletionOnlyLM(
    response_template='<|assistant|>\n',
    tokenizer=tokenizer,
)

# T4 has 16GB -- use batch_size=1, grad_accum=16 to simulate effective batch=16
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    optim='paged_adamw_32bit',
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=True,
    tf32=False,            # TF32 not available on T4
    max_grad_norm=0.3,
    report_to='none',
    dataset_text_field='text',
    max_seq_length=512,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
)

print('Starting training... (expect ~30-40 min on T4)')
trainer.train()

In [ ]:
# Step 8: Save adapter
import json
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

meta = {'model_id': MODEL_ID, 'lora_r': 16, 'lora_alpha': 32,
        'epochs': 3, 'lr': 2e-4, 'train_samples': len(train_ds)}
with open(f'{OUTPUT_DIR}/training_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Adapter saved to {OUTPUT_DIR}')
print('Download it from Kaggle: Output tab on the right > checkpoints/')

In [ ]:
# Step 9: Sanity check - confirm the model emits the action label BEFORE you download it.
# If "has action: False" prints, the run is broken. Fix and retrain, do not waste a download.
model.config.use_cache = True
model.eval()
SYS = ("Appointment assistant. Output one JSON object only. "
       "Actions: book_appointment, check_availability, cancel_appointment, clarify, out_of_scope. "
       "Services: general, consultation, follow_up. Dates: YYYY-MM-DD. Times: HH:MM. "
       'If fields missing: {"action": "clarify", "missing_fields": [...]}.')
end_id = tokenizer.convert_tokens_to_ids('<|end|>')
for t in ["Please book a consultation on Monday at 2pm.", "Do you have anything on Tuesday?"]:
    p = f"<|system|>\n{SYS}<|end|>\n<|user|>\n{t}<|end|>\n<|assistant|>\n"
    ids = tokenizer(p, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=60, do_sample=False,
                         eos_token_id=end_id, pad_token_id=tokenizer.eos_token_id)
    txt = tokenizer.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print('IN :', t); print('OUT:', txt); print('has action:', '"action"' in txt); print()